# The Support Desk

**An Agentic AI Exercise for Business Professionals**

You're a support agent starting your shift. Your inbox has tickets from customers — password resets, billing disputes, bug reports, VIP escalations, and spam. Your job: resolve as many tickets as possible while keeping customer satisfaction (CSAT) high.

**Win Condition:** Resolve 15+ tickets with CSAT >= 80% without losing all your escalation tokens.

**Rules:**
- You have **3 escalation tokens**. Wrong or unnecessary escalations cost a token. Lose all 3 = game over.
- Issuing a refund **without looking up billing history first** costs a token.
- Refunds over $100 require manager approval (escalate to billing team).
- VIP/Enterprise churn risks **must** be escalated to Account Management.
- Security concerns **must** be escalated to Security.
- Spam tickets should be closed immediately — don't waste time replying.

First, **play the game yourself** to understand the mechanics. Then implement an AI agent to do it for you.

In [ ]:
# Setup
import sys, os

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    if os.path.exists('coding-exercises'):
        !cd coding-exercises && git pull
    else:
        !git clone -b week3 https://github.com/eth-bmai-fs26/coding-exercises.git
    os.chdir('coding-exercises/the_support_desk')

sys.path.insert(0, '.')

!pip install google-genai --quiet
print('Setup complete!')

In [ ]:
from support_desk import *
print('The Support Desk loaded!')

# Preview the queue
agent, world, tools = create_game()
print()
print(world.queue_summary())

---
## Play the Game Yourself!

Use the interactive controls below to process tickets. Tips:

1. **Open a ticket** from the queue (highest priority first)
2. **Lookup** the knowledge base before taking action
3. **Reply** to the customer or **Use a template**
4. **Apply an action** if needed (reset password, issue refund, etc.)
5. **Escalate** only when the KB says you should — wrong escalations cost tokens!
6. **Resolve** the ticket when you're done

Watch your CSAT score — resolving without replying or without looking things up hurts your rating.

In [ ]:
from support_desk.interactive import play_interactive

game = play_interactive()

---
## TODO 1: Rule-Based Think Function

Now automate it! Implement a `think` function that decides which action to take each turn.

The function receives:
- `agent`: your current state (tokens, resolved count, CSAT)
- `world`: the support world (ticket queue, knowledge base, customers)
- `history`: list of previous observations, actions, and results

It must return a string like: `'ACTION: open_ticket(ticket_id="T-001")'`

**Available actions:**
- `check_queue()` — see the queue (free)
- `open_ticket(ticket_id="...")` — open a ticket
- `lookup(query="...")` — search the knowledge base
- `reply(ticket_id="...", message="...")` — reply to customer
- `apply_action(ticket_id="...", action="...")` — apply a fix
- `escalate(ticket_id="...", team="...")` — escalate to specialist
- `use_template(ticket_id="...", template="...")` — use canned response
- `resolve_ticket(ticket_id="...")` — mark as resolved
- `check_stats()` — see your stats (free)
- `take_break()` — recover 1 token (costs 5 turns)

**Strategy hints:**
- Process tickets by priority (critical first)
- Always lookup before applying actions on billing/bug tickets
- Use templates for common ticket types
- Spam tickets: just `close_spam` and resolve
- VIP/churn tickets: escalate to `account_management`
- Security tickets: escalate to `security`

In [ ]:
def think_rule_based(agent: AgentState, world: SupportWorld, history: list[dict]) -> str:
    """Decide the next action using rule-based logic.
    
    Args:
        agent: Current agent state (tokens, resolved count, CSAT).
        world: The support world (queue, tickets, KB, customers).
        history: List of dicts with observations, actions, results.
    
    Returns:
        str: An ACTION: call string.
    """
    # ========================
    # YOUR CODE HERE
    # ========================
    raise NotImplementedError('TODO: Implement think_rule_based')

In [ ]:
result = play_rule_based(think_rule_based, max_turns=100, delay=0.1)
print(f"\nResult: {'Victory!' if result['won'] else 'Not quite...'} | "
      f"Resolved: {result['resolved']} | CSAT: {result['csat']}% | "
      f"Tokens: {result['tokens']} | Turns: {result['turns']}")
print(f"Log: {result['log_file']}")

---
## TODO 2: LLM-Powered Think Function

Now use an LLM to make decisions. But remember the lessons:

1. **Autopilot** — handle obvious tickets (spam, password resets) without the LLM
2. **Deterministic scoring** — prioritize tickets by formula, not LLM judgment
3. **Memory** — summarize ticket state, don't dump raw history
4. **Guardrails** — validate LLM output before executing (block bad escalations, unverified refunds)
5. **Prompt engineering** — give the LLM clear tools, examples, and constraints

In [ ]:
import os
from google import genai

try:
    from google.colab import userdata
    api_key = userdata.get('GEMINI_API_KEY')
except (ImportError, Exception):
    api_key = os.environ.get('GEMINI_API_KEY')

client = genai.Client(api_key=api_key)
print('Gemini client ready!')

In [ ]:
def think_llm(agent: AgentState, world: SupportWorld, history: list[dict]) -> str:
    """Decide the next action using an LLM.
    
    Args:
        agent: Current agent state.
        world: The support world.
        history: Conversation history.
    
    Returns:
        str: An ACTION: call string.
    """
    # ========================
    # YOUR CODE HERE
    # ========================
    raise NotImplementedError('TODO: Implement think_llm')

In [ ]:
result = play_with_llm(
    think_fn=lambda agent, world, history: think_llm(agent, world, history),
    max_turns=100,
    delay=0.5,
)
print(f"\nResult: {'Victory!' if result['won'] else 'Not quite...'} | "
      f"Resolved: {result['resolved']} | CSAT: {result['csat']}% | "
      f"Tokens: {result['tokens']} | Turns: {result['turns']}")
print(f"Log: {result['log_file']}")

In [ ]:
# Download the game log
try:
    from google.colab import files
    files.download(result['log_file'])
except ImportError:
    print(f"Log file: {result['log_file']}")
    print('Open the file to inspect your agent\'s decisions turn by turn.')